In [1]:
!pip install crewai crewai_tools

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 6.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 6.7 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opentelemetry-exporter-otlp-proto-grpc to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 576.3/576.3 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.0/755.0 kB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 124.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.9/157.9 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38

In [2]:
!pip install -U crewai

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GOOGLE_API_KEY"] = "YOUR_GOOGLE_API_KEY"
os.environ["SERPER_API_KEY"] = "YOUR_SERPER_API_KEY"

In [4]:
from crewai import LLM
LLM=LLM(
    model="gemini/gemini-2.0-flash",
    temperature=0.4
)
print("LLM is Ready!")

LLM is Ready!


In [5]:
from crewai_tools import SerperDevTool
Search = SerperDevTool()

In [13]:
!pip install pyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 80.4 MB/s eta 0:00:00


In [16]:
import pymupdf
with pymupdf.open("/content/CBC-test-report-format-example-sample-template-Drlogy-lab-report.pdf") as doc:
    full_text = []


    for page in doc:
        text = page.get_text("text")
        full_text.append(f"{text}")
    info="\n".join(full_text)
    print(info)

Smart Pathology Laboratory
Drlogy.com
BLOOD INDICES
Packed Cell Volume (PCV)                 57.5                                  High   40 - 50                                       %
Mean Corpuscular Volume (MCV)     87.75                                           83 - 101                                    fL
MCH                                                      27.2                                             27 - 32                                       pg
MCHC                                                    32.8                                             32.5 - 34.5                                g/dL 
RDW                                                      13.6                                             11.6 - 14.0                                 %
Dr. Vimal Shah
(MD, Pathologist)
Medical Lab Technician
(DMLT, BMLT)
Dr. Payal Shah
(MD, Pathologist)
Yash M. Patel
Age : 21 Years
Sex : Male
PID : 555
Sample Collected At:
125, Shivam Bungalow, S G Road,
Mumbai
Ref. By: Dr.

In [17]:
Gender="Male"
Age="50"
Symptoms=["High Temperature","fatigue","Not able to eat","Nausea","warm Breath","Chest Pain"]
City="Lucknow"

In [18]:
from crewai import Agent, Crew
diagnoser = Agent(
    name="Diagnoser",
    role="Disease Diagnoser",
    goal=f"Diagnose a Disease or illness from {Gender}, {Age}, {Symptoms}.check for emergency red flags",
    backstory="you are a highly professional Medical Expert.you can accurately and precisely diagnose disease and illness from symptoms,age and gender",
    llm=LLM
)

In [19]:
Clinic_Suggestor = Agent(
    name="Clinic_Suggestor",
    role="Clinic_Suggestor",
    goal=f"suggest clinics based on the treatment required by patient along with the contact and address of the clinic",
    backstory="you are a highly professional Medical Expert.you can suggest good clinics for treatment required by patient on the basis of disease of the patient",
    llm=LLM
)

In [21]:
from crewai import Task
Disease_Diagnose=Task(
    name="Disease_Diagonser",
    description=f"Diagnose a Disease or illness from {Gender}, {Age}, {Symptoms}.After Diagnosing the Disease or illness also give detailed and precise cure and treatment for that disease that does not have any bad effect on the patient's body",
    expected_output="Give 2 or 3 Diagnosed disease or illness with most probability after that detailed solution for that disease",
    agent=diagnoser
)

In [22]:
Clinic_Suggestion=Task(
    name="Clinin suggestion",
    context=[Disease_Diagnose],
    description=f"With the help of {Search} Suggest 2-3 good clinics based on treatment required for the patients with good rating in {City} along with the address and contact .",
    expected_output="Give clinic name with address and contact",
    agent=Clinic_Suggestor
)

In [26]:
from crewai import Process
crew=Crew(
    agents=[diagnoser,Clinic_Suggestor],
    tasks=[Disease_Diagnose,Clinic_Suggestion]
)

In [27]:
result=crew.kickoff()
print(result)

ERROR:root:Google Gemini API error: 429 - Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/generative-ai/docs/error-code-429 for more details.


An unknown error occurred. Please check the details below.
Error details: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/generative-ai/docs/error-code-429 for more details.', 'status': 'RESOURCE_EXHAUSTED'}}
An unknown error occurred. Please check the details below.
Error details: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/generative-ai/docs/error-code-429 for more details.', 'status': 'RESOURCE_EXHAUSTED'}}
**Smart Pathology Laboratory Report Analysis**

**Patient:** Yash M. Patel
**Age:** 21 Years
**Sex:** Male

**Key Findings:**

*   **Hemoglobin (Hb):** 12.5 g/dL (Low) - Indicates anemia, a condition where the blood has a lower than normal number of red blood cells or hemoglobin.
*   **Packed Cell Volume (PCV):** 57.5% (High) - Suggests an increased concentration of

╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Would you like to view your execution traces? [y/N] (20s timeout): 